# Regression Leaf-Map Demo

This notebook demonstrates sparse train/test leaf maps for a random-forest regressor and projects them with PCA.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find the project root containing pyproject.toml and src/.")


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from forestkernel import LeafEncoder


In [ ]:
seed = 42
weight_scheme = "uniform"  # choose: "uniform", "kerf", "oob", or "gap"
calibration = "none"  # choose: "none", "global", or "componentwise"


In [ ]:
X, y = make_regression(
    n_samples=3000,
    n_features=20,
    n_informative=10,
    noise=10.0,
    random_state=seed,
)

bins = np.quantile(y, np.linspace(0, 1, 11))
y_bins = np.digitize(y, bins[1:-1])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    stratify=y_bins,
    random_state=seed,
)

print(f"Train samples: {X_train.shape[0]}; test samples: {X_test.shape[0]}")
print(f"y_train mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"y_test  mean={y_test.mean():.3f}, std={y_test.std():.3f}")


In [ ]:
forest = RandomForestRegressor(
    n_estimators=300,
    bootstrap=True,
    n_jobs=-1,
    random_state=seed,
)

encoder = LeafEncoder(forest=forest, weight_scheme=weight_scheme).fit(X_train, y_train)

leaf_train = encoder.training_query_map()
leaf_train_as_oos = encoder.transform(X_train)
leaf_test = encoder.transform(X_test)
K_test = encoder.kernel_extend(X_test)

print(f"leaf_train       : {leaf_train.shape}, nnz={leaf_train.nnz}")
print(f"leaf_train_as_oos: {leaf_train_as_oos.shape}, nnz={leaf_train_as_oos.nnz}")
print(f"leaf_test        : {leaf_test.shape}, nnz={leaf_test.nnz}")
print(f"K_test           : {K_test.shape}, nnz={K_test.nnz}")


In [ ]:
def summarize_row_norms(name, A):
    l2 = np.sqrt(A.multiply(A).sum(axis=1)).A1
    l1 = np.asarray(np.abs(A).sum(axis=1)).ravel()
    print(
        f"{name:<18} "
        f"L2 mean={l2.mean():.4f} range=({l2.min():.4f}, {l2.max():.4f}) | "
        f"L1 mean={l1.mean():.4f} range=({l1.min():.4f}, {l1.max():.4f})"
    )

summarize_row_norms("train", leaf_train)
summarize_row_norms("train as OOS", leaf_train_as_oos)
summarize_row_norms("test", leaf_test)


In [ ]:
pca = PCA(n_components=2, random_state=seed)
Z_train = pca.fit_transform(leaf_train)
Z_train_as_oos = pca.transform(leaf_train_as_oos)
Z_test_raw = pca.transform(leaf_test)

eps = 1e-12
if calibration == "none":
    Z_test = Z_test_raw
elif calibration == "global":
    numerator = np.sum(Z_train * Z_train_as_oos)
    denominator = np.sum(Z_train_as_oos ** 2)
    scale = numerator / denominator if denominator > eps else 1.0
    Z_test = scale * Z_test_raw
elif calibration == "componentwise":
    numerator = np.sum(Z_train * Z_train_as_oos, axis=0)
    denominator = np.sum(Z_train_as_oos ** 2, axis=0)
    scale = np.divide(numerator, denominator, out=np.ones_like(numerator), where=denominator > eps)
    Z_test = Z_test_raw * scale
else:
    raise ValueError(f"Unknown calibration={calibration!r}.")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
train_plot = ax.scatter(Z_train[:, 0], Z_train[:, 1], c=y_train, s=8, cmap="viridis", alpha=0.35)
ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_test, s=18, cmap="viridis", edgecolor="black", linewidth=0.3)
ax.set_title(f"Leaf-map PCA ({weight_scheme})")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.colorbar(train_plot, ax=ax, label="target")
plt.show()
